# Data Analytics — Exercise 7.3: Pandas DataFrames & Merging

## Dimitri Nji | 5/19/2026

This notebook analyzes GlobalTrade Ltd.'s order and shipment data using Pandas. It loads two CSV files, cleans the data, performs different merge types, and summarizes order fulfilment performance.

## 1. Load CSV Files with pd.read_csv()

In this section, I import Pandas and load the orders.csv and shipments.csv files. Both CSV files must be saved in the same folder as this notebook.

In [1]:
import pandas as pd
import os

print("Files in this folder:")
print(os.listdir("."))

df_orders = pd.read_csv("orders.csv")
df_shipments = pd.read_csv("shipments.csv")

print("\nOrders DataFrame:")
print(df_orders.head())
print("\nOrders shape:", df_orders.shape)
print("\nOrders data types:")
print(df_orders.dtypes)
print("\nOrders missing values:")
print(df_orders.isnull().sum())

print("\nShipments DataFrame:")
print(df_shipments.head())
print("\nShipments shape:", df_shipments.shape)
print("\nShipments data types:")
print(df_shipments.dtypes)
print("\nShipments missing values:")
print(df_shipments.isnull().sum())

Files in this folder:
['.ipynb_checkpoints', '1D, 2D, 3D Arrays.py.py', '2 Dimentional Arrays', 'DA_Exercise_7_1.ipynb', 'DA_Exercise_7_2.ipynb', 'DA_Exercise_7_3.ipynb', 'orders.csv', 'shipments.csv']

Orders DataFrame:
   order_id customer   product  quantity  unit_price
0       101     Chen    Laptop         2      899.99
1       102    Patel    Tablet         5      499.50
2       103   Müller   Monitor         1      349.00
3       104  Okonkwo  Keyboard        10       79.99
4       105   Santos   Headset         3      149.99

Orders shape: (10, 5)

Orders data types:
order_id        int64
customer          str
product           str
quantity        int64
unit_price    float64
dtype: object

Orders missing values:
order_id      0
customer      0
product       0
quantity      0
unit_price    0
dtype: int64

Shipments DataFrame:
   order_id  ship_date carrier      status
0       101  1/10/2024     DHL   Delivered
1       102  1/12/2024   FedEx   Delivered
2       103  1/13/2024    

## 2. Inspect and Clean Both DataFrames

In this section, I create an order_value column, review summary statistics, convert ship_date to datetime, and calculate the total value of all orders.

In [2]:
df_orders["order_value"] = df_orders["quantity"] * df_orders["unit_price"]

print("Orders with order value:")
print(df_orders[["order_id", "customer", "product", "quantity", "unit_price", "order_value"]])

print("\nNumeric summary statistics:")
print(df_orders.describe())

df_shipments["ship_date"] = pd.to_datetime(df_shipments["ship_date"])

print("\nShipments data types after date conversion:")
print(df_shipments.dtypes)

total_order_value = df_orders["order_value"].sum()

print(f"\nTotal order value: £{total_order_value:,.2f}")

Orders with order value:
   order_id  customer          product  quantity  unit_price  order_value
0       101      Chen           Laptop         2      899.99      1799.98
1       102     Patel           Tablet         5      499.50      2497.50
2       103    Müller          Monitor         1      349.00       349.00
3       104   Okonkwo         Keyboard        10       79.99       799.90
4       105    Santos          Headset         3      149.99       449.97
5       106   Larsson           Webcam         4       89.99       359.96
6       107       Kim            Mouse         8       29.99       239.92
7       108   O'Brien  Docking Station         1      199.00       199.00
8       109     Rossi              SSD         6      119.99       719.94
9       110  Nakamura          USB Hub         2       39.99        79.98

Numeric summary statistics:
        order_id   quantity  unit_price  order_value
count   10.00000  10.000000   10.000000    10.000000
mean   105.50000   4.20000

## 3. Performing All Four DataFrame Merge Types

In this section, I merge the orders and shipments DataFrames using inner, left, right, and outer joins. Each merge shows a different view of matched and unmatched order records.

In [3]:
df_inner = pd.merge(df_orders, df_shipments, on="order_id", how="inner")

print("INNER merge:")
print(df_inner[["order_id", "customer", "order_value", "carrier", "status"]])
print(f"\nInner merge shape: {df_inner.shape}")
print("Inner merge keeps only order IDs that appear in both DataFrames.")


df_left = pd.merge(df_orders, df_shipments, on="order_id", how="left")

print("\nLEFT merge:")
print(df_left[["order_id", "customer", "order_value", "carrier", "status"]])
print(f"\nLeft merge shape: {df_left.shape}")

unshipped = df_left[df_left["carrier"].isna()]
print("\nUnshipped orders:")
print(unshipped[["order_id", "customer", "product", "order_value"]])


df_right = pd.merge(df_orders, df_shipments, on="order_id", how="right")

print("\nRIGHT merge:")
print(df_right[["order_id", "customer", "carrier", "status"]])
print(f"\nRight merge shape: {df_right.shape}")

orphan = df_right[df_right["customer"].isna()]
print("\nOrphan shipment:")
print(orphan[["order_id", "carrier", "status"]])


df_outer = pd.merge(df_orders, df_shipments, on="order_id", how="outer")

print("\nOUTER merge:")
print(df_outer[["order_id", "customer", "carrier", "status"]])
print(f"\nOuter merge shape: {df_outer.shape}")
print(f"Total NaN values in outer merge: {df_outer.isnull().sum().sum()}")

INNER merge:
   order_id customer  order_value carrier      status
0       101     Chen      1799.98     DHL   Delivered
1       102    Patel      2497.50   FedEx   Delivered
2       103   Müller       349.00     DHL  In Transit
3       104  Okonkwo       799.90     UPS   Delivered
4       105   Santos       449.97   FedEx     Delayed
5       106  Larsson       359.96     UPS   Delivered
6       107      Kim       239.92     DHL  In Transit

Inner merge shape: (7, 9)
Inner merge keeps only order IDs that appear in both DataFrames.

LEFT merge:
   order_id  customer  order_value carrier      status
0       101      Chen      1799.98     DHL   Delivered
1       102     Patel      2497.50   FedEx   Delivered
2       103    Müller       349.00     DHL  In Transit
3       104   Okonkwo       799.90     UPS   Delivered
4       105    Santos       449.97   FedEx     Delayed
5       106   Larsson       359.96     UPS   Delivered
6       107       Kim       239.92     DHL  In Transit
7       10

## 4. Shipped Flag and Fulfilment Rate

In this section, I use the left merged DataFrame to flag each order as shipped or unshipped. Then I calculate the fulfilment rate.

In [4]:
df_left["shipped"] = df_left["carrier"].notna()

print("Shipped flag per order:")
print(df_left[["order_id", "customer", "order_value", "shipped"]])

print("\nShipped vs unshipped count:")
print(df_left["shipped"].value_counts())

shipped_count = df_left["shipped"].sum()
total_orders = len(df_left)
fulfilment_rate = shipped_count / total_orders * 100

print(f"\nFulfilment rate: {fulfilment_rate:.2f}%")

Shipped flag per order:
   order_id  customer  order_value  shipped
0       101      Chen      1799.98     True
1       102     Patel      2497.50     True
2       103    Müller       349.00     True
3       104   Okonkwo       799.90     True
4       105    Santos       449.97     True
5       106   Larsson       359.96     True
6       107       Kim       239.92     True
7       108   O'Brien       199.00    False
8       109     Rossi       719.94    False
9       110  Nakamura        79.98    False

Shipped vs unshipped count:
shipped
True     7
False    3
Name: count, dtype: int64

Fulfilment rate: 70.00%


## 5. GroupBy Summary on the Inner Merged DataFrame

In this section, I use the inner merged DataFrame to summarize order value by shipment status and carrier. Since the inner merge only includes matched records, the summary focuses only on orders that have shipment records.

In [5]:
status_summary = df_inner.groupby("status")["order_value"].agg(["sum", "mean", "count"])

print("Order value by shipment status:")
print(status_summary.round(2))

carrier_totals = df_inner.groupby("carrier")["order_value"].sum().sort_values(ascending=False)

print("\nCarrier totals:")
print(carrier_totals)

top_carrier = carrier_totals.index[0]

print(f"\nTop carrier: {top_carrier}")
print(f"Top carrier total: £{carrier_totals.iloc[0]:,.2f}")

units_per_status = df_inner.groupby("status")["quantity"].sum()

print("\nUnits dispatched per status:")
print(units_per_status)

Order value by shipment status:
                sum     mean  count
status                             
Delayed      449.97   449.97      1
Delivered   5457.34  1364.34      4
In Transit   588.92   294.46      2

Carrier totals:
carrier
FedEx    2947.47
DHL      2388.90
UPS      1159.86
Name: order_value, dtype: float64

Top carrier: FedEx
Top carrier total: £2,947.47

Units dispatched per status:
status
Delayed        3
Delivered     21
In Transit     9
Name: quantity, dtype: int64


## Summary

The fulfilment rate is 70.00%, which means 7 out of 10 customer orders have shipment records. Orders 108, 109, and 110 are unshipped because they appear in the orders file but do not have matching shipment records. FedEx is the top carrier by total order value, and order ID 111 should be reviewed because it appears in the shipments file without a matching order record.